# Hybrid QLSTM Model Implementation

## 1. Import Libraries

In [ ]:
import pennylane as qml
from pennylane import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, TimeDistributed, Layer
import os
import numpy as np

## 2. Load Data

In [ ]:
try:
    X_train = np.load('../data/X_train.npy')
    y_train = np.load('../data/y_train.npy')
    X_test = np.load('../data/X_test.npy')
    y_test = np.load('../data/y_test.npy')
    print("Data loaded.")
    print(f"X_train shape: {X_train.shape}")
except FileNotFoundError:
    print("Data files not found. Run data_prep.ipynb first.")

## 3. Define Quantum Circuit and Layer
Using Custom QuantumLayer as per Phase 3 design due to Keras 3 compatibility issues.

In [ ]:
n_qubits = 5  # Matching feature size
n_layers = 2
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def quantum_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(w)) for w in range(n_qubits)]

class QuantumLayer(Layer):
    def __init__(self, qnode, weight_shapes, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.qnode = qnode
        self.output_dim = output_dim
        self.w = self.add_weight(
            shape=weight_shapes["weights"],
            initializer="random_normal",
            trainable=True,
            name="weights"
        )

    def call(self, inputs):
        # Simple wrapper for QNode call within Keras
        return tf.convert_to_tensor(self.qnode(inputs, self.w))

    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.output_dim)

## 4. Build Hybrid Model

In [ ]:
def create_hybrid_model(input_shape):
    weight_shapes = {"weights": (n_layers, n_qubits)}
    qlayer = QuantumLayer(quantum_circuit, weight_shapes, output_dim=n_qubits)
    
    model = Sequential()
    model.add(TimeDistributed(qlayer, input_shape=input_shape))
    model.add(LSTM(50, return_sequences=True))
    model.add(LSTM(50))
    model.add(Dense(1, activation='sigmoid'))
    return model

input_shape = (60, n_qubits)
model = create_hybrid_model(input_shape)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## 5. Train Model

In [ ]:
# Warning: QML simulation is slow. Using small epochs for demonstration.
history = model.fit(X_train, y_train, epochs=1, batch_size=32, validation_data=(X_test, y_test))
print("Training Complete.")

## 6. Save Model

In [ ]:
os.makedirs('../models', exist_ok=True)
model.save('../models/qlstm_hybrid.h5')
print("Model saved to models/qlstm_hybrid.h5")